This is the data processing page where I used my Old COGS 108 conda enviroment in order to pre process data 
I got the data for the iphones via apples 10-k reports:
https://www.sec.gov/Archives/edgar/data/320193/000119312509214859/d10k.htm#toc
https://www.sec.gov/Archives/edgar/data/320193/000119312509214859/d10k.htm#toc
https://www.sec.gov/Archives/edgar/data/320193/000119312515356351/d17062d10k.htm
https://www.sec.gov/Archives/edgar/data/320193/000032019318000145/a10-k20189292018.htm

For the falling data: I used the cdc's wonder website to get people who died falling:
https://wonder.cdc.gov/controller/datarequest/D76;jsessionid=14D82FA6290F2D5FB7762DB4AE1C

The reason I went 2007 - 2018 was beacuse 2007 was when the iphone came out and 2018 was when Apple stopped reporting number of unit sales for the iphone,

Also Iphone units are per fiscal year, but should be close enough to not matter for this example

In [1]:
import pandas as pd
import numpy as np

Next Up we are going to clean up the cdc's data set, since there wasn't a dataset for the iphone sales I created my own using the links metioned before, and I also auto made my own when presented since the CDC's was unessarily complicated and it was small enough to work with


In [10]:
#loading the datset
falls_df = pd.read_csv("data/raw/falls_per_year.csv")
iphone_df = pd.read_csv("data/raw/iphone_units_sold.csv")

In [22]:
# combine them on year
df = pd.merge(iphone_df, falls_df[['year', 'deaths']], on='year', how='inner')

# min-max normalize both columns to a 0-1 scale
for col in ['iphone_units_sold_millions', 'deaths']:
    df[col + '_norm'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# save the normalized data
df = df.drop(columns=["deaths","iphone_units_sold_millions"])
df.to_csv('data/final/normalized_merged_set.csv', index=False, encoding='utf-8-sig')

# Seperating for individual line plo (for line plot iphone units sold)
iphone_norm = df[['year', 'iphone_units_sold_millions_norm']].rename(
    columns={'iphone_units_sold_millions_norm': 'iphone_norm'})
iphone_norm.to_csv('data/final/iphone_normalized.csv', index=False, encoding='utf-8-sig')

# Seperating for individual line plot (for line plot: death related to falling down the stairs)
falls_norm = df[['year', 'deaths_norm']].rename(
    columns={'deaths_norm': 'falls_norm'})
falls_norm.to_csv('data/final/falls_normalized.csv', index=False, encoding='utf-8-sig')

# combining on year, then dropping the year column (for the scatter plot)
combined = pd.merge(iphone_norm, falls_norm, on='year', how='inner')
combined = combined.drop(columns='year')
combined.to_csv('data/final/iphone_vs_falls_normalized.csv', index=False, encoding='utf-8-sig')

Now we move on to another data set and these are doozy, shark attacks that I got it's info from the global shark attack file: \
https://www.sharkattackfile.net/incidentlog.htm 

I got the data for ice cream production in the US from the federal reserves FRED database \
https://fred.stlouisfed.org/series/IPN31152N

In [12]:
#load the shark attack and ice cream dataset
shark_df = pd.read_csv("data/raw/SharkAttacks.csv")
ice_df = pd.read_csv("data/raw/IceCream.csv")

#keeping only USA rows 
shark_df['Country'] = shark_df['Country'].astype(str).str.strip().str.upper()
usa_df = shark_df[shark_df['Country'] == 'USA'].copy()

#keeping only rows from 2015 and 2016
usa_df['Year'] = pd.to_numeric(usa_df['Year'], errors='coerce')
usa_df = usa_df[usa_df['Year'].isin([2015, 2016])].reset_index(drop=True)
usa_df['Year'] = usa_df['Year'].astype(int)

#icecream is already pretty much simplified but we need to rename some stuf
#renaming columns for easy reading
ice_df = ice_df.rename(columns={'observation_date': 'date', 'IPN31152N': 'ice_cream_index'})

Now we are going to add the total amount per month, summing both years

In [13]:
# pull the month out of the Date column for sharks
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
usa_df['month'] = usa_df['Date'].astype(str).str.extract(
    r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)', expand=False)

#counting attacks per month (both years added), keeping our calendar order
monthly_df = (usa_df.groupby('month').size()
              .reindex(months, fill_value=0)
              .reset_index(name='attacks'))


#now we do the same for icecream
#this just makes it so pd sees it as a date
ice_df['date'] = pd.to_datetime(ice_df['date'])

#summing months
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ice_df['month'] = ice_df['date'].dt.strftime('%b')
ice_monthly_df = (ice_df.groupby('month')['ice_cream_index'].sum()
                  .reindex(months)
                  .round(2)
                  .reset_index())

In [14]:
#Now we normalize and combine into different files

In [15]:
# combine them on month
df = pd.merge(ice_monthly_df, monthly_df[['month', 'attacks']], on='month', how='inner')

# min-max normalize both columns to a 0-1 scale
for col in ['ice_cream_index', 'attacks']:
    df[col + '_norm'] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# save the normalized data
df = df.drop(columns=["ice_cream_index", "attacks"])
df.to_csv('data/final/normalized_icecream_sharks_merged.csv', index=False, encoding='utf-8-sig')

# Separating for individual line plot (for line plot: ice cream production)
icecream_norm = df[['month', 'ice_cream_index_norm']].rename(
    columns={'ice_cream_index_norm': 'icecream_norm'})
icecream_norm.to_csv('data/final/icecream_normalized.csv', index=False, encoding='utf-8-sig')

# Separating for individual line plot (for line plot: shark attacks)
sharks_norm = df[['month', 'attacks_norm']].rename(
    columns={'attacks_norm': 'sharks_norm'})
sharks_norm.to_csv('data/final/sharks_normalized.csv', index=False, encoding='utf-8-sig')

# combining on month, then dropping the month column (for the scatter plot)
combined = pd.merge(icecream_norm, sharks_norm, on='month', how='inner')
combined = combined.drop(columns='month')
combined.to_csv('data/final/icecream_vs_sharks_normalized.csv', index=False, encoding='utf-8-sig')